# Large-FOV Nuclear Pipeline Debug Notebook

This version is organized as **independent execution blocks** so you can rerun only the stage you want to inspect instead of rerunning the full pipeline.

## How to use this notebook

Run the notebook top to bottom the first time. After that, you can rerun only the sections you want to debug.

Typical debugging workflow:
1. **Load image**
2. **Run or load segmentation**
3. **Inspect masks / probabilities / object tables**
4. **Run or load grouping across z**
5. **Run or load best-z selection**
6. **Run or load exclusion / timing / tracking**
7. **Run or load halo analysis**
8. **Plot QC outputs**

Each stage writes intermediate artifacts to disk, so later sections can load prior results without recomputing everything.


## 1. Imports

In [1]:

from __future__ import annotations

from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
import json
import math
import os
import time

from joblib import Parallel, delayed
import numpy as np
import pandas as pd

from scipy import ndimage as ndi
from scipy.spatial import cKDTree
from scipy.optimize import linear_sum_assignment
from skimage import measure, morphology

# Optional imports
try:
    import tensorflow as tf
except Exception:
    tf = None

try:
    import tifffile as tiff
except Exception:
    tiff = None

from IPython.display import display

try:
    from sklearn.cluster import DBSCAN
except Exception:
    DBSCAN = None


2026-04-17 09:31:23.871473: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-17 09:31:23.871553: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-17 09:31:23.887999: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-17 09:31:23.953785: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-17 09:31:37.547819: W tensorflow/compiler/tf2

## 2. Configuration

In [23]:
def get_n_workers() -> int:
    return int(os.environ.get("SLURM_CPUS_PER_TASK", os.cpu_count() or 1))


@dataclass
class PipelineConfig:
    # -------------------------
    # Root paths
    # -------------------------
    project_root: str = "/home/tdeibert/Data/Machine_Learning_Dev/"
    model_subdir: str = "Models"
    model_name: str = "unet_droplet_nucleus_best_1.2.h5"
    image_subdir: str = "Images"
    input_image_name: str = "control_extract_1.1.tif"
    output_subdir: str = "Outputs"

    # -------------------------
    # Imaging
    # -------------------------
    pixel_size_um: float = 0.108
    z_step_um: float = 1.0
    nuclear_channel_index: int = 1
    membrane_channel_index: int = 0

    # -------------------------
    # Halo analysis
    # -------------------------
    halo_step_px: int = 4
    n_halos: int = 4
    erosion_px: int = 4

    # -------------------------
    # Grouping / tracking
    # -------------------------
    z_group_tolerance_um: float = 1.0
    time_track_tolerance_um: float = 5.0
    multi_nucleus_exclusion_um: float = 1.0

    track_dbscan_eps_um: float = 6.0
    track_dbscan_min_samples: int = 1
    track_crowded_distance_scale: float = 0.65
    track_area_log_weight: float = 12.0
    track_area_ratio_max: float = 5.0

    # -------------------------
    # Stitched acquisition layout
    # -------------------------
    tile_rows: int = 2
    tile_cols: int = 3
    minutes_per_tile: float = 1.0
    serpentine_scan: bool = True

    # -------------------------
    # Segmentation
    # -------------------------
    prediction_threshold: float = 0.5
    nucleus_probability_threshold: float = 0.3
    min_nucleus_area_px: int = 50
    nucleus_hole_size_px: int = 200
    nucleus_opening_radius_px: int = 1
    min_nucleus_circularity: float = 0.45
    batch_size: int = 8
    use_mixed_precision: bool = False
    patch_size: int = 512
    patch_stride: int = 256
    background_class_index: int = 0
    droplet_class_index: int = 1
    nucleus_class_index: int = 2

    # -------------------------
    # HPC
    # -------------------------
    n_workers: Optional[int] = None

    def __post_init__(self):
        if self.n_workers is None:
            self.n_workers = min(get_n_workers(), 16)

    # -------------------------
    # Derived unit conversions
    # -------------------------
    @property
    def z_group_tolerance_px(self) -> float:
        return self.z_group_tolerance_um / self.pixel_size_um

    @property
    def time_track_tolerance_px(self) -> float:
        return self.time_track_tolerance_um / self.pixel_size_um

    @property
    def multi_nucleus_exclusion_px(self) -> float:
        return self.multi_nucleus_exclusion_um / self.pixel_size_um

    @property
    def track_dbscan_eps_px(self) -> float:
        return self.track_dbscan_eps_um / self.pixel_size_um

    # -------------------------
    # Base paths
    # -------------------------
    @property
    def project_root_path(self) -> Path:
        return Path(self.project_root)

    @property
    def model_path(self) -> Path:
        return self.project_root_path / self.model_subdir / self.model_name

    @property
    def input_image_path(self) -> Path:
        return self.project_root_path / self.image_subdir / self.input_image_name

    @property
    def output_dir(self) -> Path:
        return self.project_root_path / self.output_subdir

    # -------------------------
    # Output subdirectories
    # -------------------------
    @property
    def seg_dir(self) -> Path:
        return self.output_dir / "segmentation"

    @property
    def obj_dir(self) -> Path:
        return self.output_dir / "objects"

    @property
    def track_dir(self) -> Path:
        return self.output_dir / "tracking"

    @property
    def analysis_dir(self) -> Path:
        return self.output_dir / "analysis"

    @property
    def qc_dir(self) -> Path:
        return self.output_dir / "qc"

    @property
    def mask_tif_dir(self) -> Path:
        return self.output_dir / "mask_tifs"

    # -------------------------
    # Segmentation artifact paths
    # -------------------------
    @property
    def segmentation_index_path(self) -> Path:
        return self.seg_dir / "segmentation_index.pkl"

    @property
    def segmentation_class_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_class_hyperstack.tif"

    @property
    def segmentation_label_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "segmentation_label_hyperstack.tif"

    @property
    def nucleus_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "nucleus_instance_hyperstack.tif"

    @property
    def droplet_instance_hyperstack_path(self) -> Path:
        return self.mask_tif_dir / "droplet_instance_hyperstack.tif"

    @property
    def segmentation_roi_table_path(self) -> Path:
        return self.seg_dir / "segmentation_roi_table.pkl"

    def to_serializable_dict(self) -> Dict[str, object]:
        return asdict(self)

    def __repr__(self) -> str:
        keys = [
            "project_root", "model_name", "input_image_name", "output_subdir",
            "pixel_size_um", "nuclear_channel_index", "membrane_channel_index",
            "z_group_tolerance_um", "time_track_tolerance_um",
            "multi_nucleus_exclusion_um", "track_dbscan_eps_um",
            "min_nucleus_area_px", "patch_size", "patch_stride",
            "n_workers", "use_mixed_precision",
        ]
        return "\n".join(f"{k}: {getattr(self, k)}" for k in keys)


cfg = PipelineConfig()

for p in [
    cfg.output_dir,
    cfg.seg_dir,
    cfg.obj_dir,
    cfg.track_dir,
    cfg.analysis_dir,
    cfg.qc_dir,
    cfg.mask_tif_dir,
]:
    p.mkdir(parents=True, exist_ok=True)

cfg


IndentationError: expected an indented block after class definition on line 10 (1780799671.py, line 11)

In [24]:
print(cfg.model_path)
print(cfg.input_image_path)
print(cfg.output_dir)

AttributeError: 'PipelineConfig' object has no attribute 'model_path'

## 3. Save / load configuration

In [5]:
def save_config(config: PipelineConfig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(config.to_serializable_dict(), indent=2))


def load_config(in_path: Path) -> PipelineConfig:
    data = json.loads(in_path.read_text())
    return PipelineConfig(**data)


save_config(cfg, cfg.output_dir / "pipeline_config.json")


## 4. GPU setup

In [7]:

def configure_tensorflow_for_gpu(config):
    import tensorflow as tf

    gpus = tf.config.list_physical_devices("GPU")
    print("GPUs found:", gpus)

    use_mixed_precision = getattr(config, "use_mixed_precision", False)

    if use_mixed_precision:
        try:
            from tensorflow.keras import mixed_precision
            mixed_precision.set_global_policy("mixed_float16")
            print("Mixed precision enabled")
        except Exception as e:
            print("Could not enable mixed precision:", e)

    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception as e:
            print("Could not set memory growth:", e)

configure_tensorflow_for_gpu(cfg)


GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 5. Image loading helpers

In [8]:

def load_image_5d(image_path: Union[str, Path]) -> np.ndarray:
    if tiff is None:
        raise ImportError("tifffile is required to load TIFF data.")
    arr = tiff.imread(image_path)
    print("Loaded image shape:", arr.shape)
    return arr


def get_nuclear_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.nuclear_channel_index]


def get_membrane_plane(img_5d: np.ndarray, t: int, z: int, config: PipelineConfig) -> np.ndarray:
    return img_5d[t, z, config.membrane_channel_index]


def get_full_plane_yxc(img_5d: np.ndarray, t_idx: int, z_idx: int) -> np.ndarray:
    """Return a full 3-channel plane in (Y, X, C) format from a (T, Z, C, Y, X) image."""
    plane = img_5d[t_idx, z_idx]
    return np.moveaxis(plane, 0, -1)


## 6. Segmentation helpers

In [9]:
def load_unet_model(model_path: Union[str, Path]):
    if tf is None:
        raise ImportError("TensorFlow is not available.")
    return tf.keras.models.load_model(model_path, compile=False)


def preprocess_patch_for_model(patch: np.ndarray) -> np.ndarray:
    x = patch.astype(np.float32)
    max_val = np.max(x)
    if max_val > 0:
        x = x / max_val
    return x


def generate_patch_starts(length: int, patch_size: int, stride: int) -> List[int]:
    """
    Generate patch start positions that always include the image end.
    """
    starts = list(range(0, max(length - patch_size + 1, 1), stride))
    if not starts:
        starts = [0]

    last_start = length - patch_size
    if last_start < 0:
        last_start = 0

    if starts[-1] != last_start:
        starts.append(last_start)

    return sorted(set(starts))


def extract_overlapping_patches(
    image_yxc: np.ndarray,
    patch_size: int = 512,
    stride: int = 256,
) -> Tuple[List[np.ndarray], List[Tuple[int, int]]]:
    """
    Extract overlapping patches from a preprocessed full plane.
    Returns patches and top-left coordinates.
    """
    h, w, c = image_yxc.shape
    y_starts = generate_patch_starts(h, patch_size, stride)
    x_starts = generate_patch_starts(w, patch_size, stride)

    patches: List[np.ndarray] = []
    coords: List[Tuple[int, int]] = []

    for y0 in y_starts:
        for x0 in x_starts:
            patch = image_yxc[y0:y0 + patch_size, x0:x0 + patch_size, :]
            patches.append(patch)
            coords.append((y0, x0))

    return patches, coords


def predict_patch_probabilities(model, batch_patches: np.ndarray) -> np.ndarray:
    """
    Predict raw class probabilities for a batch of patches.
    Returns shape (B, H, W, C).
    """
    preds = model.predict(batch_patches, verbose=0)

    if preds.ndim != 4:
        raise ValueError(f"Unexpected prediction shape: {preds.shape}")

    if preds.shape[-1] == 1:
        fg = preds[..., 0]
        bg = 1.0 - fg
        preds = np.stack([bg, fg], axis=-1)

    return preds.astype(np.float32)


def stitch_probability_patches(
    patch_probs: List[np.ndarray],
    coords: List[Tuple[int, int]],
    image_shape: Tuple[int, int],
    n_classes: int,
    patch_size: int = 512,
) -> np.ndarray:
    """
    Stitch overlapping patch probabilities by averaging.
    Returns shape (H, W, C).
    """
    h, w = image_shape
    prob_sum = np.zeros((h, w, n_classes), dtype=np.float32)
    prob_count = np.zeros((h, w, 1), dtype=np.float32)

    for probs, (y0, x0) in zip(patch_probs, coords):
        y1 = y0 + patch_size
        x1 = x0 + patch_size
        prob_sum[y0:y1, x0:x1, :] += probs
        prob_count[y0:y1, x0:x1, :] += 1.0

    prob_map = prob_sum / np.maximum(prob_count, 1.0)
    return prob_map


def filter_labeled_objects_by_area_and_circularity(
    binary_mask: np.ndarray,
    min_area_px: int = 50,
    min_circularity: float = 0.45,
) -> np.ndarray:
    """
    Filter connected components by area and circularity.
    """
    labeled = measure.label(binary_mask, connectivity=2)
    filtered_mask = np.zeros_like(binary_mask, dtype=bool)

    for prop in measure.regionprops(labeled):
        area = float(prop.area)
        perimeter = float(prop.perimeter)

        if area < min_area_px:
            continue

        if perimeter <= 0:
            continue

        circularity = (4.0 * np.pi * area) / (perimeter ** 2)

        if circularity >= min_circularity:
            filtered_mask[labeled == prop.label] = True

    return filtered_mask


def clean_nucleus_mask(
    nucleus_mask: np.ndarray,
    min_size_px: int = 50,
    hole_size_px: int = 200,
    opening_radius: int = 1,
    min_circularity: float = 0.45,
) -> np.ndarray:
    mask = nucleus_mask.astype(bool)
    mask = morphology.remove_small_objects(mask, min_size=min_size_px)
    mask = morphology.remove_small_holes(mask, area_threshold=hole_size_px)

    if opening_radius > 0:
        mask = morphology.binary_opening(mask, footprint=morphology.disk(opening_radius))

    mask = ndi.binary_fill_holes(mask)

    mask = filter_labeled_objects_by_area_and_circularity(
        binary_mask=mask,
        min_area_px=min_size_px,
        min_circularity=min_circularity,
    )

    return mask.astype(bool)


def segment_single_plane_with_overlap(
    plane_yxc: np.ndarray,
    model,
    config: PipelineConfig,
    patch_size: int = 512,
    stride: int = 256,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Segment one full plane using overlapping patches and probability stitching.

    Returns
    -------
    class_prob_map : (Y, X, C)
    class_label_map : (Y, X)
    nucleus_mask : (Y, X) bool
    nucleus_instance_labels : (Y, X) uint16
    """
    h, w, c = plane_yxc.shape

    patches, coords = extract_overlapping_patches(
        plane_yxc,
        patch_size=patch_size,
        stride=stride,
    )
    patches = [preprocess_patch_for_model(p) for p in patches]

    patch_probs: List[np.ndarray] = []

    for i in range(0, len(patches), config.batch_size):
        batch = np.stack(patches[i:i + config.batch_size], axis=0)
        batch_pred = predict_patch_probabilities(model, batch)
        patch_probs.extend(batch_pred)

    n_classes = patch_probs[0].shape[-1]

    class_prob_map = stitch_probability_patches(
        patch_probs=patch_probs,
        coords=coords,
        image_shape=(h, w),
        n_classes=n_classes,
        patch_size=patch_size,
    )

    class_label_map = np.argmax(class_prob_map, axis=-1).astype(np.uint8)
    nucleus_prob = class_prob_map[..., config.nucleus_class_index]
    nucleus_mask = nucleus_prob > config.nucleus_probability_threshold

    nucleus_mask = clean_nucleus_mask(
        nucleus_mask,
        min_size_px=config.min_nucleus_area_px,
        hole_size_px=config.nucleus_hole_size_px,
        opening_radius=config.nucleus_opening_radius_px,
        min_circularity=config.min_nucleus_circularity,
    )

    nucleus_instance_labels = measure.label(nucleus_mask, connectivity=2).astype(np.uint16)

    return class_prob_map, class_label_map, nucleus_mask, nucleus_instance_labels


def build_class_binary_stack(
    class_label_map: np.ndarray,
    config: PipelineConfig,
) -> np.ndarray:
    return np.stack(
        [
            class_label_map == config.background_class_index,
            class_label_map == config.droplet_class_index,
            class_label_map == config.nucleus_class_index,
        ],
        axis=0,
    ).astype(np.uint8)


def save_class_hyperstack_tiff(class_mask_5d: np.ndarray, out_path: Path) -> None:
    if tiff is None:
        raise ImportError("tifffile is required to save TIFF data.")
    tiff.imwrite(
        out_path,
        (class_mask_5d.astype(np.uint8) * 255),
        imagej=True,
        metadata={"axes": "TZCYX"},
    )


def save_label_hyperstack_tiff(label_4d: np.ndarray, out_path: Path) -> None:
    if tiff is None:
        raise ImportError("tifffile is required to save TIFF data.")
    tiff.imwrite(
        out_path,
        label_4d.astype(np.uint8),
        imagej=True,
        metadata={"axes": "TZYX"},
    )


def save_instance_hyperstack_tiff(instance_4d: np.ndarray, out_path: Path) -> None:
    if tiff is None:
        raise ImportError("tifffile is required to save TIFF data.")
    tiff.imwrite(
        out_path,
        instance_4d,
        imagej=True,
        metadata={"axes": "TZYX"},
    )


def regionprops_to_rows(
    labeled_mask: np.ndarray,
    t_idx: int,
    z_idx: int,
    class_id: int,
    class_name: str,
) -> List[Dict[str, Union[int, float, str]]]:
    rows: List[Dict[str, Union[int, float, str]]] = []
    for prop in measure.regionprops(labeled_mask):
        cy, cx = prop.centroid
        perimeter = float(prop.perimeter)
        circularity = np.nan
        if perimeter > 0:
            circularity = (4.0 * np.pi * float(prop.area)) / (perimeter ** 2)

        rows.append(
            {
                "t": t_idx,
                "z": z_idx,
                "class_id": class_id,
                "class_name": class_name,
                "label": int(prop.label),
                "centroid_x_px": float(cx),
                "centroid_y_px": float(cy),
                "area_px": int(prop.area),
                "perimeter_px": perimeter,
                "circularity": circularity,
                "bbox_min_row": int(prop.bbox[0]),
                "bbox_min_col": int(prop.bbox[1]),
                "bbox_max_row": int(prop.bbox[2]),
                "bbox_max_col": int(prop.bbox[3]),
            }
        )
    return rows


def run_segmentation_for_all_planes(
    img_5d: np.ndarray,
    model,
    config: PipelineConfig,
    save_binary_masks: bool = True,
) -> pd.DataFrame:
    records = []
    roi_rows: List[Dict[str, Union[int, float, str]]] = []
    T, Z = img_5d.shape[0], img_5d.shape[1]
    Y, X = img_5d.shape[-2], img_5d.shape[-1]

    class_mask_5d = np.zeros((T, Z, 3, Y, X), dtype=np.uint8)
    class_label_4d = np.zeros((T, Z, Y, X), dtype=np.uint8)
    nucleus_instance_4d = np.zeros((T, Z, Y, X), dtype=np.uint16)
    droplet_instance_4d = np.zeros((T, Z, Y, X), dtype=np.uint16)

    for t_idx in range(T):
        for z_idx in range(Z):
            plane_yxc = get_full_plane_yxc(img_5d, t_idx, z_idx)

            class_prob_map, class_label_map, nucleus_mask, nucleus_instance_labels = segment_single_plane_with_overlap(
                plane_yxc=plane_yxc,
                model=model,
                config=config,
                patch_size=config.patch_size,
                stride=config.patch_stride,
            )

            background_mask = class_label_map == config.background_class_index
            droplet_mask = class_label_map == config.droplet_class_index

            class_mask_5d[t_idx, z_idx, 0] = background_mask.astype(np.uint8)
            class_mask_5d[t_idx, z_idx, 1] = droplet_mask.astype(np.uint8)
            class_mask_5d[t_idx, z_idx, 2] = nucleus_mask.astype(np.uint8)

            class_label_4d[t_idx, z_idx] = class_label_map.astype(np.uint8)

            droplet_instances = measure.label(droplet_mask, connectivity=2).astype(np.uint16)

            nucleus_instance_4d[t_idx, z_idx] = nucleus_instance_labels.astype(np.uint16)
            droplet_instance_4d[t_idx, z_idx] = droplet_instances

            roi_rows.extend(
                regionprops_to_rows(
                    labeled_mask=nucleus_instance_labels,
                    t_idx=t_idx,
                    z_idx=z_idx,
                    class_id=config.nucleus_class_index,
                    class_name="nucleus",
                )
            )
            roi_rows.extend(
                regionprops_to_rows(
                    labeled_mask=droplet_instances,
                    t_idx=t_idx,
                    z_idx=z_idx,
                    class_id=config.droplet_class_index,
                    class_name="droplet",
                )
            )

            out_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
            if save_binary_masks:
                np.save(out_path, nucleus_mask.astype(np.uint8))

            records.append({
                "t": t_idx,
                "z": z_idx,
                "mask_path": str(out_path),
            })

    save_class_hyperstack_tiff(class_mask_5d, config.segmentation_class_hyperstack_path)
    save_label_hyperstack_tiff(class_label_4d, config.segmentation_label_hyperstack_path)
    save_instance_hyperstack_tiff(nucleus_instance_4d, config.nucleus_instance_hyperstack_path)
    save_instance_hyperstack_tiff(droplet_instance_4d, config.droplet_instance_hyperstack_path)

    roi_df = pd.DataFrame(roi_rows)
    roi_df.to_pickle(config.segmentation_roi_table_path)

    seg_index_df = pd.DataFrame(records)
    seg_index_df.to_pickle(config.segmentation_index_path)
    return seg_index_df

## 7. Object extraction from masks

In [10]:

def label_and_measure_objects(mask: np.ndarray, t_idx: int, z_idx: int, config: PipelineConfig) -> pd.DataFrame:
    labeled = measure.label(mask, connectivity=2)
    props = measure.regionprops(labeled)

    rows = []
    for prop in props:
        if prop.area < config.min_nucleus_area_px:
            continue

        cy, cx = prop.centroid
        rows.append({
            "t": t_idx,
            "z": z_idx,
            "label": int(prop.label),
            "centroid_x_px": float(cx),
            "centroid_y_px": float(cy),
            "area_px": int(prop.area),
            "bbox_min_row": int(prop.bbox[0]),
            "bbox_min_col": int(prop.bbox[1]),
            "bbox_max_row": int(prop.bbox[2]),
            "bbox_max_col": int(prop.bbox[3]),
        })

    return pd.DataFrame(rows)


def extract_objects_from_saved_masks(seg_index_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    all_rows = []

    for row in seg_index_df.itertuples(index=False):
        mask = np.load(row.mask_path).astype(bool)
        obj_df = label_and_measure_objects(mask, row.t, row.z, config)
        all_rows.append(obj_df)

    objects_df = pd.concat(all_rows, ignore_index=True) if all_rows else pd.DataFrame()
    objects_df.to_pickle(config.obj_dir / "plane_objects.pkl")
    return objects_df


## 8. Distance-based helpers

In [11]:

def euclidean_distance_px(x1: float, y1: float, x2: float, y2: float) -> float:
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def nearest_neighbor_matches(
    source_df: pd.DataFrame,
    target_df: pd.DataFrame,
    max_dist_px: float,
) -> List[Tuple[int, int, float]]:
    if source_df.empty or target_df.empty:
        return []

    source_xy = source_df[["centroid_x_px", "centroid_y_px"]].to_numpy()
    target_xy = target_df[["centroid_x_px", "centroid_y_px"]].to_numpy()

    tree = cKDTree(target_xy)
    dists, idxs = tree.query(source_xy, distance_upper_bound=max_dist_px)

    matches = []
    used_targets = set()

    order = np.argsort(dists)
    for i in order:
        dist = dists[i]
        j = idxs[i]
        if np.isinf(dist):
            continue
        if j in used_targets:
            continue
        matches.append((int(i), int(j), float(dist)))
        used_targets.add(int(j))

    return matches


## 9. Group nuclei across z

In [12]:

def group_nuclei_across_z(objects_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if objects_df.empty:
        return pd.DataFrame()

    grouped_rows = []
    next_group_id = 1

    for t_idx, df_t in objects_df.groupby("t", sort=True):
        df_t = df_t.sort_values(["z", "label"]).reset_index(drop=True).copy()
        df_t["nucleus_3d_id"] = -1
        z_values = sorted(df_t["z"].unique())

        first_z = z_values[0]
        first_idx = df_t.index[df_t["z"] == first_z].tolist()
        for idx in first_idx:
            df_t.at[idx, "nucleus_3d_id"] = next_group_id
            next_group_id += 1

        for z_prev, z_curr in zip(z_values[:-1], z_values[1:]):
            prev_df = df_t[df_t["z"] == z_prev].reset_index()
            curr_df = df_t[df_t["z"] == z_curr].reset_index()

            matches = nearest_neighbor_matches(prev_df, curr_df, config.z_group_tolerance_px)
            matched_curr_global = set()

            for i_prev, i_curr, dist in matches:
                prev_global_idx = int(prev_df.loc[i_prev, "index"])
                curr_global_idx = int(curr_df.loc[i_curr, "index"])
                matched_curr_global.add(curr_global_idx)

                prev_group = df_t.at[prev_global_idx, "nucleus_3d_id"]
                if prev_group == -1:
                    prev_group = next_group_id
                    df_t.at[prev_global_idx, "nucleus_3d_id"] = prev_group
                    next_group_id += 1

                df_t.at[curr_global_idx, "nucleus_3d_id"] = prev_group

            for curr_global_idx in curr_df["index"].tolist():
                if curr_global_idx not in matched_curr_global and df_t.at[curr_global_idx, "nucleus_3d_id"] == -1:
                    df_t.at[curr_global_idx, "nucleus_3d_id"] = next_group_id
                    next_group_id += 1

        grouped_rows.append(df_t)

    grouped_z_df = pd.concat(grouped_rows, ignore_index=True)
    grouped_z_df.to_pickle(config.obj_dir / "grouped_z_objects.pkl")
    return grouped_z_df


## 10. Select best z per nucleus

In [13]:
def select_best_z_per_nucleus(grouped_z_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if grouped_z_df.empty:
        return pd.DataFrame()

    best_z_df = (
        grouped_z_df
        .sort_values(["t", "nucleus_3d_id", "area_px"], ascending=[True, True, False])
        .groupby(["t", "nucleus_3d_id"], as_index=False)
        .head(1)
        .reset_index(drop=True)
    )

    best_z_df.to_pickle(config.obj_dir / "best_z_nuclei.pkl")
    return best_z_df

## 11. Exclude likely multi-nucleus droplets

In [14]:

def flag_close_nuclei(best_z_df: pd.DataFrame, config: PipelineConfig) -> pd.DataFrame:
    if best_z_df.empty:
        return pd.DataFrame()

    out = []
    for t_idx, df_t in best_z_df.groupby("t", sort=True):
        df_t = df_t.copy().reset_index(drop=True)
        coords = df_t[["centroid_x_px", "centroid_y_px"]].to_numpy()
        tree = cKDTree(coords)
        close_flags = np.zeros(len(df_t), dtype=bool)

        for i, xy in enumerate(coords):
            neighbor_idx = tree.query_ball_point(xy, r=config.multi_nucleus_exclusion_px)
            neighbor_idx = [j for j in neighbor_idx if j != i]
            if len(neighbor_idx) > 0:
                close_flags[i] = True
                for j in neighbor_idx:
                    close_flags[j] = True

        df_t["valid_single_nucleus"] = ~close_flags
        out.append(df_t)

    filtered_df = pd.concat(out, ignore_index=True)
    filtered_df.to_pickle(config.obj_dir / "best_z_nuclei_with_exclusion.pkl")
    return filtered_df


## 12. Tile assignment and true acquisition time

In [15]:

def assign_tile_index_from_xy(
    x_px: float,
    y_px: float,
    image_width_px: int,
    image_height_px: int,
    config: PipelineConfig,
) -> Tuple[int, int, int]:
    tile_w = image_width_px / config.tile_cols
    tile_h = image_height_px / config.tile_rows

    col = min(int(x_px // tile_w), config.tile_cols - 1)
    row = min(int(y_px // tile_h), config.tile_rows - 1)

    if config.serpentine_scan:
        if row % 2 == 0:
            tile_index = row * config.tile_cols + col
        else:
            tile_index = row * config.tile_cols + (config.tile_cols - 1 - col)
    else:
        tile_index = row * config.tile_cols + col

    return row, col, tile_index

def add_tile_timing_metadata(nuclei_df: pd.DataFrame, img_5d: np.ndarray, config: PipelineConfig) -> pd.DataFrame:
    if nuclei_df.empty:
        return pd.DataFrame()

    image_height_px = img_5d.shape[-2]
    image_width_px = img_5d.shape[-1]

    out_rows = []
    for row in nuclei_df.itertuples(index=False):
        tile_row, tile_col, tile_index = assign_tile_index_from_xy(
            x_px=row.centroid_x_px,
            y_px=row.centroid_y_px,
            image_width_px=image_width_px,
            image_height_px=image_height_px,
            config=config
        )

        tile_offset_min = tile_index * config.minutes_per_tile
        true_time_min = row.t * (config.tile_rows * config.tile_cols * config.minutes_per_tile) + tile_offset_min

        d = row._asdict()
        d["tile_row"] = tile_row
        d["tile_col"] = tile_col
        d["tile_index"] = tile_index
        d["tile_offset_min"] = tile_offset_min
        d["true_time_min"] = true_time_min
        out_rows.append(d)

    timed_df = pd.DataFrame(out_rows)
    timed_df.to_pickle(config.track_dir / "best_z_nuclei_timed.pkl")
    return timed_df


## 13. Track nuclei across time

This section now uses a **hybrid DBSCAN-assisted tracker**:

1. build local spatial neighborhoods with DBSCAN from consecutive frames,
2. score candidate links by distance plus area consistency,
3. use one-to-one assignment inside each local neighborhood,
4. reject crowded or low-confidence links rather than forcing a track.


In [16]:

def _safe_area_ratio(area_a: float, area_b: float) -> float:
    area_a = max(float(area_a), 1.0)
    area_b = max(float(area_b), 1.0)
    return max(area_a, area_b) / min(area_a, area_b)


def _link_cost(prev_row: pd.Series, curr_row: pd.Series, distance_px: float, config: PipelineConfig) -> float:
    area_ratio = _safe_area_ratio(prev_row["area_px"], curr_row["area_px"])
    area_penalty = config.track_area_log_weight * abs(math.log(area_ratio))
    return float(distance_px + area_penalty)


def build_dbscan_candidate_clusters(
    prev_df: pd.DataFrame,
    curr_df: pd.DataFrame,
    config: PipelineConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if DBSCAN is None:
        raise ImportError("scikit-learn is required for DBSCAN-assisted tracking.")

    prev_local = prev_df.copy().reset_index(drop=True)
    curr_local = curr_df.copy().reset_index(drop=True)

    prev_local["frame_role"] = "prev"
    curr_local["frame_role"] = "curr"
    prev_local["frame_local_index"] = np.arange(len(prev_local))
    curr_local["frame_local_index"] = np.arange(len(curr_local))

    pooled = pd.concat([prev_local, curr_local], ignore_index=True)
    if pooled.empty:
        pooled["dbscan_cluster_id"] = pd.Series(dtype=int)
        pooled["dbscan_cluster_size"] = pd.Series(dtype=int)
        pooled["dbscan_is_crowded"] = pd.Series(dtype=bool)
        return prev_local, curr_local, pooled

    coords = pooled[["centroid_x_px", "centroid_y_px"]].to_numpy()
    labels = DBSCAN(
        eps=config.track_dbscan_eps_px,
        min_samples=config.track_dbscan_min_samples,
    ).fit_predict(coords)

    pooled["dbscan_cluster_id"] = labels

    cluster_sizes = pooled.groupby("dbscan_cluster_id")["frame_local_index"].transform("size")
    pooled["dbscan_cluster_size"] = cluster_sizes.astype(int)

    crowded_flags = []
    for cluster_id, df_cluster in pooled.groupby("dbscan_cluster_id", sort=False):
        n_prev = int((df_cluster["frame_role"] == "prev").sum())
        n_curr = int((df_cluster["frame_role"] == "curr").sum())
        crowded_flags.extend([n_prev > 1 or n_curr > 1] * len(df_cluster))
    pooled["dbscan_is_crowded"] = crowded_flags

    prev_out = pooled[pooled["frame_role"] == "prev"].copy().reset_index(drop=True)
    curr_out = pooled[pooled["frame_role"] == "curr"].copy().reset_index(drop=True)
    return prev_out, curr_out, pooled


def match_cluster_with_assignment(
    prev_cluster: pd.DataFrame,
    curr_cluster: pd.DataFrame,
    config: PipelineConfig,
) -> List[Dict[str, Union[int, float, bool]]]:
    if prev_cluster.empty or curr_cluster.empty:
        return []

    crowded = bool(
        prev_cluster["dbscan_is_crowded"].any()
        or curr_cluster["dbscan_is_crowded"].any()
        or len(prev_cluster) > 1
        or len(curr_cluster) > 1
    )

    max_dist_px = config.time_track_tolerance_px
    if crowded:
        max_dist_px = max_dist_px * config.track_crowded_distance_scale

    prev_xy = prev_cluster[["centroid_x_px", "centroid_y_px"]].to_numpy()
    curr_xy = curr_cluster[["centroid_x_px", "centroid_y_px"]].to_numpy()

    cost_matrix = np.full((len(prev_cluster), len(curr_cluster)), np.inf, dtype=float)
    dist_matrix = np.full((len(prev_cluster), len(curr_cluster)), np.inf, dtype=float)

    for i, prev_row in prev_cluster.reset_index(drop=True).iterrows():
        for j, curr_row in curr_cluster.reset_index(drop=True).iterrows():
            dist_px = euclidean_distance_px(
                prev_row["centroid_x_px"],
                prev_row["centroid_y_px"],
                curr_row["centroid_x_px"],
                curr_row["centroid_y_px"],
            )
            if dist_px > max_dist_px:
                continue

            area_ratio = _safe_area_ratio(prev_row["area_px"], curr_row["area_px"])
            if area_ratio > config.track_area_ratio_max:
                continue

            cost_matrix[i, j] = _link_cost(prev_row, curr_row, dist_px, config)
            dist_matrix[i, j] = dist_px

    if not np.isfinite(cost_matrix).any():
        return []

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    matches: List[Dict[str, Union[int, float, bool]]] = []

    for i, j in zip(row_ind, col_ind):
        cost_val = cost_matrix[i, j]
        dist_val = dist_matrix[i, j]
        if not np.isfinite(cost_val):
            continue

        matches.append({
            "prev_local_index": int(prev_cluster.iloc[i]["index"]),
            "curr_local_index": int(curr_cluster.iloc[j]["index"]),
            "link_distance_px": float(dist_val),
            "link_cost": float(cost_val),
            "dbscan_cluster_id": int(prev_cluster.iloc[i]["dbscan_cluster_id"]),
            "dbscan_cluster_size": int(prev_cluster.iloc[i]["dbscan_cluster_size"]),
            "dbscan_is_crowded": bool(crowded),
        })

    return matches


def assign_track_ids_hybrid_dbscan(
    timed_df: pd.DataFrame,
    config: PipelineConfig,
    valid_only: bool = False,
) -> pd.DataFrame:
    if timed_df.empty:
        return pd.DataFrame()

    df = timed_df.copy()
    if valid_only and "valid_single_nucleus" in df.columns:
        df = df[df["valid_single_nucleus"]].copy()

    df = df.sort_values(["t", "nucleus_3d_id"]).reset_index(drop=True)
    df["track_id"] = -1
    df["track_link_distance_px"] = np.nan
    df["track_link_cost"] = np.nan
    df["track_dbscan_cluster_id"] = -1
    df["track_dbscan_cluster_size"] = 0
    df["track_dbscan_is_crowded"] = False
    df["track_link_method"] = "unassigned"

    time_values = sorted(df["t"].unique())
    if not time_values:
        return df

    next_track_id = 1
    first_time = time_values[0]
    idx0 = df.index[df["t"] == first_time].tolist()
    for idx in idx0:
        df.at[idx, "track_id"] = next_track_id
        df.at[idx, "track_link_method"] = "seed"
        next_track_id += 1

    for t_prev, t_curr in zip(time_values[:-1], time_values[1:]):
        prev_df = df[df["t"] == t_prev].reset_index()
        curr_df = df[df["t"] == t_curr].reset_index()

        prev_clustered, curr_clustered, pooled = build_dbscan_candidate_clusters(prev_df, curr_df, config)

        cluster_matches: List[Dict[str, Union[int, float, bool]]] = []
        for cluster_id in sorted(pooled["dbscan_cluster_id"].unique()):
            prev_cluster = prev_clustered[prev_clustered["dbscan_cluster_id"] == cluster_id].copy().reset_index(drop=True)
            curr_cluster = curr_clustered[curr_clustered["dbscan_cluster_id"] == cluster_id].copy().reset_index(drop=True)
            cluster_matches.extend(match_cluster_with_assignment(prev_cluster, curr_cluster, config))

        matched_curr_global = set()

        for match in sorted(cluster_matches, key=lambda x: x["link_cost"]):
            prev_global_idx = int(match["prev_local_index"])
            curr_global_idx = int(match["curr_local_index"])

            if curr_global_idx in matched_curr_global:
                continue

            track_id = int(df.at[prev_global_idx, "track_id"])
            if track_id == -1:
                track_id = next_track_id
                df.at[prev_global_idx, "track_id"] = track_id
                df.at[prev_global_idx, "track_link_method"] = "backfilled_seed"
                next_track_id += 1

            matched_curr_global.add(curr_global_idx)
            df.at[curr_global_idx, "track_id"] = track_id
            df.at[curr_global_idx, "track_link_distance_px"] = float(match["link_distance_px"])
            df.at[curr_global_idx, "track_link_cost"] = float(match["link_cost"])
            df.at[curr_global_idx, "track_dbscan_cluster_id"] = int(match["dbscan_cluster_id"])
            df.at[curr_global_idx, "track_dbscan_cluster_size"] = int(match["dbscan_cluster_size"])
            df.at[curr_global_idx, "track_dbscan_is_crowded"] = bool(match["dbscan_is_crowded"])
            df.at[curr_global_idx, "track_link_method"] = "hybrid_dbscan"

        for _, row in curr_clustered.iterrows():
            curr_global_idx = int(row["index"])
            if curr_global_idx in matched_curr_global:
                continue

            df.at[curr_global_idx, "track_dbscan_cluster_id"] = int(row["dbscan_cluster_id"])
            df.at[curr_global_idx, "track_dbscan_cluster_size"] = int(row["dbscan_cluster_size"])
            df.at[curr_global_idx, "track_dbscan_is_crowded"] = bool(row["dbscan_is_crowded"])

            if df.at[curr_global_idx, "track_id"] == -1:
                df.at[curr_global_idx, "track_id"] = next_track_id
                df.at[curr_global_idx, "track_link_method"] = "new_track_unmatched"
                next_track_id += 1

    df.to_pickle(config.track_dir / "tracked_nuclei.pkl")
    return df


def summarize_tracking_debug(tracked_df: pd.DataFrame) -> pd.DataFrame:
    if tracked_df.empty:
        return pd.DataFrame()

    summary = {
        "n_rows": int(len(tracked_df)),
        "n_tracks": int(tracked_df["track_id"].nunique()),
        "median_track_length": float(tracked_df.groupby("track_id").size().median()),
        "mean_track_length": float(tracked_df.groupby("track_id").size().mean()),
        "fraction_crowded_links": float(tracked_df["track_dbscan_is_crowded"].fillna(False).mean()),
        "fraction_new_tracks": float((tracked_df["track_link_method"] == "new_track_unmatched").mean()),
    }
    return pd.DataFrame([summary])


## 14. Fiji-equivalent cumulative halos

In [17]:

def build_cumulative_halo_masks(nucleus_mask: np.ndarray, config: PipelineConfig) -> Dict[str, np.ndarray]:
    dist_outside = ndi.distance_transform_edt(~nucleus_mask)
    cumulative = {"nucleus_mask": nucleus_mask.astype(bool)}

    for i in range(1, config.n_halos + 1):
        max_dist = i * config.halo_step_px
        cumulative[f"halo_{i}_cum"] = dist_outside <= max_dist

    for i in range(1, config.n_halos + 1):
        if i == 1:
            cumulative[f"ring_{i}"] = cumulative["halo_1_cum"] & (~nucleus_mask)
        else:
            cumulative[f"ring_{i}"] = cumulative[f"halo_{i}_cum"] & (~cumulative[f"halo_{i-1}_cum"])

    if config.erosion_px > 0:
        eroded = ndi.binary_erosion(nucleus_mask, structure=morphology.disk(config.erosion_px))
        cumulative["nucleus_eroded"] = eroded
    else:
        cumulative["nucleus_eroded"] = nucleus_mask.copy()

    cumulative["cytoplasm_mask"] = cumulative[f"halo_{config.n_halos}_cum"] & (~nucleus_mask)
    return cumulative

def mask_stats(intensity_image: np.ndarray, mask: np.ndarray) -> Dict[str, float]:
    area_px = int(mask.sum())
    if area_px == 0:
        return {"area_px": 0, "intden": 0.0, "mean_intensity": np.nan}
    vals = intensity_image[mask]
    return {
        "area_px": area_px,
        "intden": float(vals.sum()),
        "mean_intensity": float(vals.mean()),
    }

def measure_fiji_equivalent_halos(intensity_image: np.ndarray, nucleus_mask: np.ndarray, config: PipelineConfig) -> Dict[str, float]:
    masks = build_cumulative_halo_masks(nucleus_mask, config)
    out = {}

    nuc = mask_stats(intensity_image, masks["nucleus_mask"])
    for k, v in nuc.items():
        out[f"nucleus_{k}"] = v

    nuc_eroded = mask_stats(intensity_image, masks["nucleus_eroded"])
    for k, v in nuc_eroded.items():
        out[f"nucleus_eroded_{k}"] = v

    for i in range(1, config.n_halos + 1):
        stats = mask_stats(intensity_image, masks[f"halo_{i}_cum"])
        for k, v in stats.items():
            out[f"halo_{i}_cum_{k}"] = v

    for i in range(1, config.n_halos + 1):
        stats = mask_stats(intensity_image, masks[f"ring_{i}"])
        for k, v in stats.items():
            out[f"ring_{i}_{k}"] = v

    cyto = mask_stats(intensity_image, masks["cytoplasm_mask"])
    for k, v in cyto.items():
        out[f"cytoplasm_{k}"] = v

    nuclear_mean = out["nucleus_mean_intensity"]
    cytoplasm_mean = out["cytoplasm_mean_intensity"]

    out["nc_ratio"] = float(nuclear_mean / cytoplasm_mean) if np.isfinite(nuclear_mean) and np.isfinite(cytoplasm_mean) and cytoplasm_mean != 0 else np.nan
    out["nc_ratio_fraction"] = float(nuclear_mean / (nuclear_mean + cytoplasm_mean)) if np.isfinite(nuclear_mean) and np.isfinite(cytoplasm_mean) and (nuclear_mean + cytoplasm_mean) != 0 else np.nan
    return out


## 15. Recover a nucleus mask for a tracked object

In [18]:

def recover_nucleus_mask_from_plane(
    t_idx: int,
    z_idx: int,
    centroid_x_px: float,
    centroid_y_px: float,
    config: PipelineConfig,
) -> np.ndarray:
    mask_path = config.seg_dir / f"nuclear_mask_t{t_idx:03d}_z{z_idx:03d}.npy"
    mask = np.load(mask_path).astype(bool)

    labeled = measure.label(mask, connectivity=2)
    props = measure.regionprops(labeled)

    best_label = None
    best_dist = np.inf

    for prop in props:
        cy, cx = prop.centroid
        dist = euclidean_distance_px(cx, cy, centroid_x_px, centroid_y_px)
        if dist < best_dist:
            best_dist = dist
            best_label = prop.label

    if best_label is None:
        return np.zeros_like(mask, dtype=bool)

    return labeled == best_label


## 16. Run halo analysis for tracked nuclei

In [19]:

def _process_single_tracked_nucleus(row, img_5d: np.ndarray, config: PipelineConfig) -> dict:
    t_idx = int(row.t)
    z_idx = int(row.z)
    cx = float(row.centroid_x_px)
    cy = float(row.centroid_y_px)

    nucleus_mask = recover_nucleus_mask_from_plane(t_idx, z_idx, cx, cy, config)
    nuclear_plane = get_nuclear_plane(img_5d, t_idx, z_idx, config)

    halo_metrics = measure_fiji_equivalent_halos(
        intensity_image=nuclear_plane,
        nucleus_mask=nucleus_mask,
        config=config,
    )

    rec = row._asdict()
    rec.update(halo_metrics)
    rec["nucleus_area_um2"] = rec["nucleus_area_px"] * (config.pixel_size_um ** 2)
    rec["cytoplasm_area_um2"] = rec["cytoplasm_area_px"] * (config.pixel_size_um ** 2)
    return rec


def run_halo_analysis_for_tracked_nuclei(
    tracked_df: pd.DataFrame,
    img_5d: np.ndarray,
    config: PipelineConfig,
) -> pd.DataFrame:
    if tracked_df.empty:
        halo_df = pd.DataFrame()
        halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
        return halo_df

    rows = Parallel(
        n_jobs=max(1, int(config.n_workers)),
        backend="threading",
        batch_size=1,
    )(
        delayed(_process_single_tracked_nucleus)(row, img_5d, config)
        for row in tracked_df.itertuples(index=False)
    )

    halo_df = pd.DataFrame(rows)
    halo_df.to_pickle(config.analysis_dir / "halo_analysis.pkl")
    return halo_df


## 17. Radial sweep placeholder

In [20]:

def radial_sweep_membrane_placeholder(
    membrane_plane: np.ndarray,
    nucleus_mask: np.ndarray,
    centroid_x_px: float,
    centroid_y_px: float,
    n_angles: int = 180,
    max_radius_px: int = 200,
) -> pd.DataFrame:
    rows = []
    angles = np.linspace(0, 360, n_angles, endpoint=False)
    for angle_deg in angles:
        rows.append({
            "angle_deg": float(angle_deg),
            "peak_intensity": np.nan,
            "peak_radius_px": np.nan,
        })
    return pd.DataFrame(rows)


## 18. Debug execution controls

In [21]:

# Set any stage to True when you want to recompute it from the previous stage.
# Set it to False to load the saved artifact from disk instead.

RUN_SEGMENTATION = True
RUN_OBJECT_EXTRACTION = True
RUN_GROUP_ACROSS_Z = True
RUN_SELECT_BEST_Z = True
RUN_FLAG_CLOSE_NUCLEI = True
RUN_ADD_TIMING = True
RUN_TRACKING = True
RUN_HALO_ANALYSIS = True

# Optional quick-debug controls
DEBUG_T_IDX = 0
DEBUG_Z_IDX = 0

stage_flags = {
    "segmentation": RUN_SEGMENTATION,
    "object_extraction": RUN_OBJECT_EXTRACTION,
    "group_across_z": RUN_GROUP_ACROSS_Z,
    "select_best_z": RUN_SELECT_BEST_Z,
    "flag_close_nuclei": RUN_FLAG_CLOSE_NUCLEI,
    "add_timing": RUN_ADD_TIMING,
    "tracking": RUN_TRACKING,
    "halo_analysis": RUN_HALO_ANALYSIS,
}
stage_flags


{'segmentation': True,
 'object_extraction': True,
 'group_across_z': True,
 'select_best_z': True,
 'flag_close_nuclei': True,
 'add_timing': True,
 'tracking': True,
 'halo_analysis': True}

## 19. Initialize runtime and confirm paths

In [22]:

configure_tensorflow_for_gpu(cfg)

print(f"Model path: {cfg.model_path}")
print(f"Input image path: {cfg.input_image_path}")
print(f"Output directory: {cfg.output_dir}")

assert cfg.model_path.exists(), f"Missing model: {cfg.model_path}"
assert cfg.input_image_path.exists(), f"Missing image: {cfg.input_image_path}"

save_config(cfg, cfg.output_dir / "debug_pipeline_config.json")
print("Configuration saved to disk.")


GPUs found: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


AttributeError: 'PipelineConfig' object has no attribute 'model_path'

## 20. Load image once

In [ ]:

t0 = time.perf_counter()
img_5d = load_image_5d(cfg.input_image_path)
print(f"Loaded image shape: {img_5d.shape}")
print(f"Image dtype: {img_5d.dtype}")
print(f"Image load time: {time.perf_counter() - t0:.2f} sec")


## 21. Optional single-plane segmentation debug

In [ ]:

model = None

if RUN_SEGMENTATION:
    model = load_unet_model(cfg.model_path)
    print("Model loaded for segmentation.")

plane_yxc = get_full_plane_yxc(img_5d, DEBUG_T_IDX, DEBUG_Z_IDX)
print(f"Debug plane shape: {plane_yxc.shape}")

if model is not None:
    debug_prob_map, debug_class_map, debug_nucleus_mask, debug_instance_labels = segment_single_plane_with_overlap(
        plane_yxc=plane_yxc,
        model=model,
        config=cfg,
        patch_size=cfg.patch_size,
        stride=cfg.patch_stride,
    )
    print("Single-plane debug segmentation complete.")
    print(f"Probability map shape: {debug_prob_map.shape}")
    print(f"Unique class labels: {np.unique(debug_class_map)}")
    print(f"Nucleus pixels: {int(debug_nucleus_mask.sum())}")


## 22. Run or load full segmentation

In [ ]:

if RUN_SEGMENTATION:
    if model is None:
        model = load_unet_model(cfg.model_path)

    t0 = time.perf_counter()
    seg_index_df = run_segmentation_for_all_planes(img_5d=img_5d, model=model, config=cfg)
    print(f"Segmentation finished in {time.perf_counter() - t0:.2f} sec")
else:
    seg_index_df = pd.read_pickle(cfg.segmentation_index_path)
    print("Loaded saved segmentation index.")

print(seg_index_df.shape)
display(seg_index_df.head())


## 23. Run or load object extraction

In [ ]:

objects_path = cfg.obj_dir / "plane_objects.pkl"

if RUN_OBJECT_EXTRACTION:
    t0 = time.perf_counter()
    objects_df = extract_objects_from_saved_masks(seg_index_df, cfg)
    print(f"Object extraction finished in {time.perf_counter() - t0:.2f} sec")
else:
    objects_df = pd.read_pickle(objects_path)
    print("Loaded saved object table.")

print(objects_df.shape)
display(objects_df.head())


## 24. Run or load grouping across z

In [ ]:

grouped_z_path = cfg.obj_dir / "grouped_z_objects.pkl"

if RUN_GROUP_ACROSS_Z:
    t0 = time.perf_counter()
    grouped_z_df = group_nuclei_across_z(objects_df, cfg)
    print(f"Grouping across z finished in {time.perf_counter() - t0:.2f} sec")
else:
    grouped_z_df = pd.read_pickle(grouped_z_path)
    print("Loaded grouped-z table.")

print(grouped_z_df.shape)
display(grouped_z_df.head())


## 25. Run or load best-z selection

In [ ]:

best_z_path = cfg.obj_dir / "best_z_nuclei.pkl"

if RUN_SELECT_BEST_Z:
    t0 = time.perf_counter()
    best_z_df = select_best_z_per_nucleus(grouped_z_df, cfg)
    print(f"Best-z selection finished in {time.perf_counter() - t0:.2f} sec")
else:
    best_z_df = pd.read_pickle(best_z_path)
    print("Loaded best-z table.")

print(best_z_df.shape)
display(best_z_df.head())


## 26. Optional area-based cleanup for debugging

In [ ]:
def filter_nuclei_by_area(df: pd.DataFrame,
                         min_area_um2: float = 100,
                         max_area_um2: float = 300) -> pd.DataFrame:
    """
    Remove objects that are too small or too large to be nuclei.
    """
    df = df.copy()

    if "nucleus_area_um2" not in df.columns:
        df["nucleus_area_um2"] = df["area_px"] * (cfg.pixel_size_um ** 2)

    filtered = df[
        (df["nucleus_area_um2"] >= min_area_um2) &
        (df["nucleus_area_um2"] <= max_area_um2)
    ]

    print(f"Filtered: {len(df)} → {len(filtered)} nuclei")

    return filtered

best_z_area_filtered_df = filter_nuclei_by_area(best_z_df, min_area_um2=100, max_area_um2=300)
display(best_z_area_filtered_df.head())

## 27. Run or load close-nuclei exclusion

In [ ]:

filtered_path = cfg.obj_dir / "best_z_nuclei_with_exclusion.pkl"

if RUN_FLAG_CLOSE_NUCLEI:
    t0 = time.perf_counter()
    filtered_df = flag_close_nuclei(best_z_df, cfg)
    print(f"Close-nuclei flagging finished in {time.perf_counter() - t0:.2f} sec")
else:
    filtered_df = pd.read_pickle(filtered_path)
    print("Loaded exclusion table.")

print(filtered_df.shape)
display(filtered_df.head())


## 28. Run or load timing metadata

In [ ]:

timed_path = cfg.track_dir / "best_z_nuclei_timed.pkl"

if RUN_ADD_TIMING:
    t0 = time.perf_counter()
    timed_df = add_tile_timing_metadata(filtered_df, img_5d, cfg)
    print(f"Timing metadata finished in {time.perf_counter() - t0:.2f} sec")
else:
    timed_df = pd.read_pickle(timed_path)
    print("Loaded timed nuclei table.")

print(timed_df.shape)
display(timed_df.head())


## 29. Run or load tracking

In [ ]:

tracked_path = cfg.track_dir / "tracked_nuclei.pkl"

if RUN_TRACKING:
    t0 = time.perf_counter()
    tracked_df = assign_track_ids_hybrid_dbscan(timed_df, cfg, valid_only=False)
    print(f"Hybrid DBSCAN tracking finished in {time.perf_counter() - t0:.2f} sec")
else:
    tracked_df = pd.read_pickle(tracked_path)
    print("Loaded tracked nuclei table.")

print(tracked_df.shape)
display(tracked_df.head())
display(summarize_tracking_debug(tracked_df))


## 30. Run or load halo analysis

In [ ]:

halo_path = cfg.analysis_dir / "halo_analysis.pkl"

if RUN_HALO_ANALYSIS:
    t0 = time.perf_counter()
    halo_df = run_halo_analysis_for_tracked_nuclei(tracked_df, img_5d, cfg)
    print(f"Halo analysis finished in {time.perf_counter() - t0:.2f} sec")
else:
    halo_df = pd.read_pickle(halo_path)
    print("Loaded halo analysis table.")

print(halo_df.shape)
display(halo_df.head())


## 31. QC plotting helpers

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


def plot_nucleus_mask_overlay_qc(
    img_5d: np.ndarray,
    nucleus_mask_4d: np.ndarray,
    config,
    timepoints=None,
    z_mode: str = "middle",
    max_cols: int = 4
) -> None:
    """
    QC overlay plot for nucleus segmentation.

    Parameters
    ----------
    img_5d : np.ndarray
        Raw image with shape (T, Z, C, Y, X)
    nucleus_mask_4d : np.ndarray
        Binary nucleus mask hyperstack with shape (T, Z, Y, X)
    config : PipelineConfig
        Uses config.nuclear_channel_index
    timepoints : list[int] or None
        Which timepoints to show. If None, choose evenly spaced timepoints.
    z_mode : str
        "middle" or "max_mask"
        - "middle": use middle z slice
        - "max_mask": use z slice with largest nucleus mask area at each timepoint
    max_cols : int
        Number of columns in figure layout
    """
    T, Z, C, Y, X = img_5d.shape

    if timepoints is None:
        n_show = min(T, 6)
        timepoints = np.linspace(0, T - 1, n_show, dtype=int).tolist()

    n_panels = len(timepoints)
    n_cols = min(max_cols, n_panels)
    n_rows = int(np.ceil(n_panels / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = np.atleast_1d(axes).ravel()

    for ax, t_idx in zip(axes, timepoints):
        if z_mode == "middle":
            z_idx = Z // 2
        elif z_mode == "max_mask":
            z_sums = nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1)
            z_idx = int(np.argmax(z_sums))
        else:
            raise ValueError("z_mode must be 'middle' or 'max_mask'")

        raw_plane = img_5d[t_idx, z_idx, config.nuclear_channel_index].astype(np.float32)
        mask_plane = nucleus_mask_4d[t_idx, z_idx].astype(bool)

        if raw_plane.max() > 0:
            raw_plane = raw_plane / raw_plane.max()

        ax.imshow(raw_plane, cmap="gray")
        ax.imshow(mask_plane, alpha=0.35)
        ax.set_title(f"t={t_idx}, z={z_idx}")
        ax.axis("off")

    for ax in axes[n_panels:]:
        ax.axis("off")

    fig.suptitle("QC: Raw nuclear channel with nucleus mask overlay", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import label, find_contours


def plot_nucleus_mask_qc_summary(
    nucleus_mask_4d: np.ndarray,
    timepoints=None,
    z_mode: str = "max_mask",
    max_cols: int = 4
) -> None:
    """
    QC summary for binary nucleus masks.

    Top: selected mask snapshots with object outlines
    Bottom: total segmented nuclear area vs time
    """
    T, Z, Y, X = nucleus_mask_4d.shape

    if timepoints is None:
        n_show = min(T, 6)
        timepoints = np.linspace(0, T - 1, n_show, dtype=int).tolist()

    n_panels = len(timepoints)
    n_cols = min(max_cols, n_panels)
    n_rows = int(np.ceil(n_panels / n_cols))

    fig = plt.figure(figsize=(4 * n_cols, 4 * (n_rows + 1)))
    gs = fig.add_gridspec(n_rows + 1, n_cols)

    panel_axes = []
    for r in range(n_rows):
        for c in range(n_cols):
            panel_axes.append(fig.add_subplot(gs[r, c]))

    for ax, t_idx in zip(panel_axes, timepoints):
        if z_mode == "middle":
            z_idx = Z // 2
        elif z_mode == "max_mask":
            z_sums = nucleus_mask_4d[t_idx].reshape(Z, -1).sum(axis=1)
            z_idx = int(np.argmax(z_sums))
        else:
            raise ValueError("z_mode must be 'middle' or 'max_mask'")

        mask_plane = nucleus_mask_4d[t_idx, z_idx].astype(bool)
        labeled = label(mask_plane)

        ax.imshow(mask_plane, cmap="gray")

        for contour in find_contours(mask_plane.astype(float), level=0.5):
            ax.plot(contour[:, 1], contour[:, 0], linewidth=0.5)

        ax.set_title(f"t={t_idx}, z={z_idx}\nobjects={labeled.max()}")
        ax.axis("off")

    for ax in panel_axes[n_panels:]:
        ax.axis("off")

    area_ax = fig.add_subplot(gs[n_rows, :])
    total_area_px = nucleus_mask_4d.reshape(T, -1).sum(axis=1)
    area_ax.plot(np.arange(T), total_area_px, marker="o")
    area_ax.set_xlabel("Time index")
    area_ax.set_ylabel("Total nucleus mask area (px)")
    area_ax.set_title("QC: Total segmented nuclear area over time")

    plt.tight_layout()
    plt.show()

In [ ]:
from skimage.measure import label
import numpy as np
import matplotlib.pyplot as plt


def plot_nucleus_count_over_time(nucleus_mask_4d: np.ndarray) -> None:
    """
    Count connected nucleus objects across all z slices for each timepoint.
    """
    T, Z, Y, X = nucleus_mask_4d.shape
    counts = []

    for t_idx in range(T):
        count_t = 0
        for z_idx in range(Z):
            count_t += label(nucleus_mask_4d[t_idx, z_idx].astype(bool)).max()
        counts.append(count_t)

    plt.figure(figsize=(7, 4))
    plt.plot(np.arange(T), counts, marker="o")
    plt.xlabel("Time index")
    plt.ylabel("Connected nucleus objects")
    plt.title("QC: Connected nucleus object count over time")
    plt.tight_layout()
    plt.show()

In [ ]:

import matplotlib.pyplot as plt

def plot_nc_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot.")
        return

    plt.figure(figsize=(7, 4))
    for track_id, df_track in halo_df.groupby("track_id"):
        df_track = df_track.sort_values("true_time_min")
        plt.plot(df_track["true_time_min"], df_track["nc_ratio_fraction"], marker="o", label=f"Track {track_id}")
    plt.xlabel("True acquisition time (min)")
    plt.ylabel("N / (N + C)")
    plt.title("N/C ratio fraction by track")
    plt.tight_layout()
    plt.show()

def plot_area_vs_time(halo_df: pd.DataFrame) -> None:
    if halo_df.empty:
        print("No data to plot.")
        return

    plt.figure(figsize=(7, 4))
    for track_id, df_track in halo_df.groupby("track_id"):
        df_track = df_track.sort_values("true_time_min")
        plt.plot(df_track["true_time_min"], df_track["nucleus_area_um2"], marker="o", label=f"Track {track_id}")
    plt.xlabel("True acquisition time (min)")
    plt.ylabel("Nuclear area (µm²)")
    plt.title("Largest nuclear cross-sectional area by track")
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_nucleus_probability_qc(
    class_prob_map: np.ndarray,
    raw_plane: np.ndarray,
    title=""
):
    """
    class_prob_map: (Y, X, C)
    """
    nucleus_prob = class_prob_map[..., 2]

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 3, 1)
    plt.imshow(raw_plane, cmap="gray")
    plt.title("Raw")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(nucleus_prob, cmap="inferno")
    plt.title("Nucleus probability")
    plt.colorbar()

    plt.subplot(1, 3, 3)
    plt.hist(nucleus_prob.ravel(), bins=100)
    plt.title("Probability distribution")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd


def plot_largest_cross_sectional_area_vs_time(best_z_df: pd.DataFrame) -> None:
    """
    Plot the largest cross-sectional nuclear area for all nuclei as a function of time.

    Expects best_z_df to contain:
        - true_time_min
        - area_px
        or nucleus_area_um2

    If nucleus_area_um2 is not present, it will be computed from area_px
    using cfg.pixel_size_um.
    """
    if best_z_df.empty:
        print("No data to plot.")
        return

    plot_df = best_z_df.copy()

    if "nucleus_area_um2" not in plot_df.columns:
        if "area_px" not in plot_df.columns:
            raise ValueError("best_z_df must contain either 'nucleus_area_um2' or 'area_px'.")
        plot_df["nucleus_area_um2"] = plot_df["area_px"] * (cfg.pixel_size_um ** 2)

    if "true_time_min" not in plot_df.columns:
        raise ValueError("best_z_df must contain 'true_time_min'.")

    plot_df = plot_df.sort_values("true_time_min")

    mean_df = (
        plot_df.groupby("true_time_min", as_index=False)["nucleus_area_um2"]
        .mean()
        .rename(columns={"nucleus_area_um2": "mean_nucleus_area_um2"})
    )

    plt.figure(figsize=(8, 5))
    plt.scatter(
        plot_df["true_time_min"],
        plot_df["nucleus_area_um2"],
        alpha=0.35,
        s=20
    )
    plt.plot(
        mean_df["true_time_min"],
        mean_df["mean_nucleus_area_um2"],
        linewidth=2
    )
    plt.xlabel("True acquisition time (min)")
    plt.ylabel("Largest cross-sectional nuclear area (µm²)")
    plt.title("Largest cross-sectional nuclear area vs time")
    plt.tight_layout()
    plt.show()

## 32. Load saved mask hyperstacks for QC

In [ ]:

class_mask_5d = tiff.imread(cfg.segmentation_class_hyperstack_path)
label_mask_4d = tiff.imread(cfg.segmentation_label_hyperstack_path)
nucleus_instance_4d = tiff.imread(cfg.nucleus_instance_hyperstack_path)

print(f"class_mask_5d shape: {class_mask_5d.shape}")
print(f"label_mask_4d shape: {label_mask_4d.shape}")
print(f"nucleus_instance_4d shape: {nucleus_instance_4d.shape}")

nucleus_mask_4d = class_mask_5d[:, :, cfg.nucleus_class_index] > 0
print(f"nucleus_mask_4d shape: {nucleus_mask_4d.shape}")


## 33. QC plots

In [ ]:

plot_nucleus_mask_overlay_qc(
    img_5d=img_5d,
    nucleus_mask_4d=nucleus_mask_4d,
    config=cfg,
    timepoints=None,
    z_mode="max_mask",
)

plot_nucleus_mask_qc_summary(
    nucleus_mask_4d=nucleus_mask_4d,
    timepoints=None,
    z_mode="max_mask",
)

plot_nucleus_count_over_time(nucleus_mask_4d)


## 34. Plot downstream measurements

In [ ]:

plot_nc_vs_time(halo_df)
plot_area_vs_time(halo_df)
plot_largest_cross_sectional_area_vs_time(best_z_df)


## 35. Probability-map debug on the selected plane

In [ ]:

if "debug_prob_map" in globals():
    raw_plane = img_5d[DEBUG_T_IDX, DEBUG_Z_IDX, cfg.nuclear_channel_index]
    plot_nucleus_probability_qc(
        class_prob_map=debug_prob_map,
        raw_plane=raw_plane,
        title=f"t={DEBUG_T_IDX}, z={DEBUG_Z_IDX}",
    )
else:
    print("Run the single-plane segmentation debug cell first.")


## 36. Handy table summaries

In [ ]:

summary = {
    "seg_index_rows": len(seg_index_df),
    "objects_rows": len(objects_df),
    "grouped_z_rows": len(grouped_z_df),
    "best_z_rows": len(best_z_df),
    "filtered_rows": len(filtered_df),
    "timed_rows": len(timed_df),
    "tracked_rows": len(tracked_df),
    "halo_rows": len(halo_df),
}

summary_df = pd.DataFrame([summary])
display(summary_df)


## 37. Notes

- This notebook intentionally avoids the single `run_full_pipeline(...)` wrapper.
- Every major stage is exposed as its own runnable block.
- Intermediate tables are saved to disk so you can rerun only the stage you are debugging.